# Rule or shortcut: replication runs (Bhattamishra et al. 2023 sparse parity)

Runtime > Change runtime type > T4 GPU, then Run all. Logs are written to `/content/out` on the runtime, and each run prints a one-line summary below. Set `USE_DRIVE = True` to also keep logs in Google Drive (needs a Drive permission click).

In [ ]:
import os
USE_DRIVE = False
OUT = '/content/out'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/rule-or-shortcut'
os.makedirs(OUT, exist_ok=True)

In [ ]:
!git clone -q https://github.com/anjun-c/transformer-simplicity /content/ts
!mkdir -p /content/shims

In [ ]:
# Compatibility shims: attrdict breaks on Python 3.10+, and we skip ipdb and wandb
open('/content/shims/attrdict.py','w').write('''class AttrDict(dict):
    def __getattr__(self, k):
        try: return self[k]
        except KeyError: raise AttributeError(k)
    def __setattr__(self, k, v): self[k] = v
''')
open('/content/shims/ipdb.py','w').write('from pdb import *\n')
open('/content/shims/wandb.py','w').write('def init(*a,**k): pass\ndef log(*a,**k): pass\n')

In [ ]:
import re
def parse(path):
    ep, tr, va, cur = [], [], [], {}
    for line in open(path):
        m = re.search(r'Epoch:\s*(\d+)', line)
        if m: cur = {'ep': int(m.group(1))}
        m = re.search(r'train acc:\s*([\d.]+)', line)
        if m and 'ep' in cur: cur['tr'] = float(m.group(1))
        m = re.search(r'val acc epoch:\s*([\d.]+)', line)
        if m and 'ep' in cur:
            ep.append(cur['ep']); tr.append(cur.get('tr')); va.append(float(m.group(1))); cur = {}
    return ep, tr, va

def summary(name):
    ep, tr, va = parse(f'{OUT}/{name}.log')
    if not va:
        print('RESULT', name, 'no epochs logged; tail:', open(f'{OUT}/{name}.log').read()[-500:]); return
    g = next((e for e, v in zip(ep, va) if v > 0.9), None)
    t = next((e for e, v in zip(ep, tr) if v and v > 0.99), None)
    curve = ' '.join(f'{e}:{v:.2f}' for e, v in zip(ep, va) if e % 10 == 0)
    print(f'RESULT {name} | last epoch {ep[-1]} | final train {tr[-1]:.4f} | max val {max(va):.4f} | first val>0.9 at {g} | first train>0.99 at {t}')
    print(f'VALCURVE {name} | {curve}')


In [ ]:
import subprocess, os
def run(model_dir, dataset, name, extra):
    env = dict(os.environ, PYTHONPATH='/content/shims')
    cmd = f'cd /content/ts/{model_dir} && python -m src.main -mode train -gpu 0 -dataset {dataset} -run_name {name} {extra}'
    with open(f'{OUT}/{name}.log', 'w') as f:
        subprocess.run(cmd, shell=True, env=env, stdout=f, stderr=subprocess.STDOUT)
    summary(name)

TRAFO = '-depth 4 -lr 0.001'
# Noisy labels: 5% flipped, 40k training examples (the paper's noisy setting)
run('FLTAtt', 'sparse_n_parity4a', 'trafo_noisy_s1729', TRAFO + ' -epochs 300')
run('FLTClassifier', 'sparse_n_parity4a', 'lstm_noisy_s1729', '-epochs 300')

In [ ]:
# Clean data, more seeds
for s in [1, 2, 3, 4]:
    run('FLTAtt', 'sparity40_5k', f'trafo_s{s}', TRAFO + f' -epochs 400 -seed {s}')
    run('FLTClassifier', 'sparity40_5k', f'lstm_s{s}', f'-epochs 400 -seed {s}')

In [ ]:
import re, glob
import matplotlib.pyplot as plt
def parse(path):
    ep, tr, va, cur = [], [], [], {}
    for line in open(path):
        m = re.search(r'Epoch:\s*(\d+)', line)
        if m: cur = {'ep': int(m.group(1))}
        m = re.search(r'train acc:\s*([\d.]+)', line)
        if m and 'ep' in cur: cur['tr'] = float(m.group(1))
        m = re.search(r'val acc epoch:\s*([\d.]+)', line)
        if m and 'ep' in cur:
            ep.append(cur['ep']); tr.append(cur.get('tr')); va.append(float(m.group(1))); cur = {}
    return ep, tr, va
for path in sorted(glob.glob(f'{OUT}/*.log')):
    ep, tr, va = parse(path)
    if va: print(os.path.basename(path), 'epochs', ep[-1], 'final train', round(tr[-1],4), 'max val', round(max(va),4))